# Sequence Trend Joint Warm-Start Official Eval v1

Purpose: repair the previous metric-source mismatch by letting official qlib replay start before the evaluation window, then slicing and rebasing the daily path back to the requested walk-forward test window. This tests whether the joint BC candidate was rejected only because the prior official replay started cold at the fold boundary.

Governance:
- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1`.
- Direct comparison: `sequence_trend_joint_bc_screen_v1` and `sequence_trend_joint_metric_reconciliation_v1`.
- Role: diagnose.
- Changed layer: `model_selection` / `harness_metric_diagnostic`; reusable warm-start helpers live in `src/soxl_quant/core/metric_reconciliation.py`.
- Gate: every tested fold needs relative return >= +30pp and MaxDD <= 15% on official warm-start daily paths.
- External references: Qlib official backtest stack (`https://github.com/microsoft/qlib`), Qlib paper (`https://arxiv.org/abs/2009.11189`), DAgger imitation-learning caution (`https://proceedings.mlr.press/v15/ross11a.html`), and purged CV / leakage discipline (`https://doi.org/10.21314/JCF.2016.322`).

Historical reflection:
- Closest success: `trend_participation_bc_warmstart_screen_v1` improved trend capture but did not solve all fold drawdown gates.
- Closest failure: `sequence_trend_joint_metric_reconciliation_v1` found split/official metric mismatch and blocked PPO handoff.
- This is not a duplicate because it reruns official qlib replay with warm-start state and then slices evaluation metrics, instead of reinterpreting already cold-started daily paths.

In [1]:

from __future__ import annotations

import json
import math
import os
import random
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "AGENTS.md").exists():
    ROOT = Path("/Users/rziiiii/.openclaw/workspace")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_conditioned_objective import daily_path_metrics
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.core.metric_reconciliation import (
    build_warm_start_replay_segment,
    slice_warm_start_daily_path,
)
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, flatten_feature_frame, init_qlib
from src.soxl_quant.settings import LOOKBACK, QLIB_DATA_PATH

patch_gym_compat()

EXP = "sequence_trend_joint_warm_start_official_eval_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXP
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_CARD = ROOT / "runtime" / "model_selection" / "card" / "sequence_trend_joint_bc_screen_v1"
RECON_CARD = ROOT / "runtime" / "model_selection" / "card" / "sequence_trend_joint_metric_reconciliation_v1"
FRAME_PATH = ROOT / "data" / "soxl_profile_balanced_rule_residual_v22.pkl"
PROFILE_NAME = "sparse_release_bc_ppo_screen_v1"
BASE_TARGETS_PATH = ROOT / "runtime/model_selection/card/cap10_sequence_risk_target_cap_bc_screen_v1/baseline_no_sequence_cap_target_contract.csv"
TREND_TARGETS_PATH = ROOT / "runtime/model_selection/card/trend_participation_foldlocal_target_v1/selected_trend_foldlocal_targets.csv"
JOINT_SLOPE_TARGETS_PATH = SOURCE_CARD / "trend_plus_slope_low_cap15_target_contract.csv"
SEED = 20260526
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
BC_TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=300, lr=0.001, batch_size=128, seed=SEED)
WALK_FORWARD_WINDOWS = {
    "wf_2022": {"target_fold": "wf_2022_train", "train": ("2010-03-11", "2020-12-31"), "valid": ("2021-01-01", "2021-12-31"), "test": ("2022-01-01", "2022-12-31")},
    "wf_2023_2024": {"target_fold": "wf_2023_2024_train", "train": ("2010-03-11", "2022-12-31"), "valid": ("2023-01-01", "2023-12-31"), "test": ("2024-01-01", "2024-12-31")},
}
CANDIDATES = [
    {"candidate": "base_plain_contract", "target_path": BASE_TARGETS_PATH, "complexity": 0, "sample_weight_config": None},
    {"candidate": "trend_plain_contract", "target_path": TREND_TARGETS_PATH, "complexity": 1, "sample_weight_config": None},
    {
        "candidate": "trend_plus_slope_low_cap15_weighted",
        "target_path": JOINT_SLOPE_TARGETS_PATH,
        "complexity": 3,
        "sample_weight_config": StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=3.0,
            downtrend_multiplier=2.0,
            ordinary_multiplier=1.0,
            objective_weight_power=0.20,
            class_balanced_beta=0.99,
            weight_cap=25.0,
        ),
    },
]
EXTERNAL_SOURCES = [
    "https://github.com/microsoft/qlib",
    "https://arxiv.org/abs/2009.11189",
    "https://proceedings.mlr.press/v15/ross11a.html",
    "https://doi.org/10.21314/JCF.2016.322",
]


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, np.floating):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_targets(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, parse_dates=["datetime"])
    for col in ("is_rebound_cluster", "is_downtrend_state"):
        if col not in frame.columns:
            frame[col] = False
        frame[col] = frame[col].fillna(False).astype(bool)
    for col in ("teacher_position", "rebound_target_position", "risk_target_position", "objective_weight"):
        if col not in frame.columns:
            frame[col] = 0.0
        frame[col] = pd.to_numeric(frame[col], errors="coerce").fillna(0.0)
    return frame.sort_values(["fold", "datetime"]).reset_index(drop=True)


def fold_target_subset(targets: pd.DataFrame, spec: dict[str, Any]) -> pd.DataFrame:
    train_end = pd.Timestamp(spec["train"][1])
    subset = targets[
        targets["fold"].astype(str).eq(str(spec["target_fold"]))
        & (pd.to_datetime(targets["datetime"]) <= train_end)
    ].copy()
    return subset.sort_values("datetime").reset_index(drop=True)


def gate_loss(relative_return: float, max_dd: float) -> float:
    return float(max(0.0, TARGET_RELATIVE_RETURN - relative_return) + 10.0 * max(0.0, max_dd - TARGET_MAX_DD))

source_selection = json.loads((SOURCE_CARD / "selection.json").read_text(encoding="utf-8"))
recon_selection = json.loads((RECON_CARD / "selection.json").read_text(encoding="utf-8"))
profile = load_profile(PROFILE_NAME)
train_frame = pd.read_pickle(FRAME_PATH)
targets_by_candidate = {item["candidate"]: load_targets(item["target_path"]) for item in CANDIDATES}
init_qlib(QLIB_DATA_PATH)
feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
print("source decision", source_selection.get("decision"))
print("metric reconciliation decision", recon_selection.get("decision"))
print("lookback", LOOKBACK)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[33437:MainThread](2026-05-26 16:34:26,120) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[33437:MainThread](2026-05-26 16:34:26,720) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[33437:MainThread](2026-05-26 16:34:26,721) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


source decision sequence_trend_joint_bc_mixed_no_promotion
metric reconciliation decision sequence_trend_joint_metric_mismatch_no_promotion
lookback 60


In [2]:

# Build fold-local full feature frames once. The official replay can start before
# the evaluation window, while metrics are sliced back to the requested test seg.
fold_cache: dict[str, dict[str, Any]] = {}
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    dataset = build_dataset(
        train_seg=spec["train"],
        valid_seg=spec["valid"],
        test_seg=spec["test"],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    feature_full = flatten_feature_frame(dataset.handler.feature_frame).droplevel("instrument").copy()
    price_full = dataset.handler.price_frame.droplevel("instrument")["price"].sort_index()
    warm_seg = build_warm_start_replay_segment(feature_full.index, eval_seg=spec["test"], lookback=LOOKBACK)
    fold_cache[fold_name] = {
        "dataset": dataset,
        "feature_full": feature_full,
        "price_full": price_full,
        "warm_seg": warm_seg,
    }

cache_summary = pd.DataFrame([
    {
        "fold": fold,
        "feature_start": str(cache["feature_full"].index.min().date()),
        "feature_end": str(cache["feature_full"].index.max().date()),
        "test_start": WALK_FORWARD_WINDOWS[fold]["test"][0],
        "test_end": WALK_FORWARD_WINDOWS[fold]["test"][1],
        "warm_start": cache["warm_seg"][0],
        "warm_end": cache["warm_seg"][1],
    }
    for fold, cache in fold_cache.items()
])
cache_summary.to_csv(CARD / "warm_start_segments.csv", index=False)
display(cache_summary)


[33437:MainThread](2026-05-26 16:34:28,976) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Loading data Done


[33437:MainThread](2026-05-26 16:34:28,978) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[33437:MainThread](2026-05-26 16:34:28,980) INFO - qlib.timer - [log.py:127] - Time cost: 0.007s | Init data Done


[33437:MainThread](2026-05-26 16:34:31,433) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[33437:MainThread](2026-05-26 16:34:31,434) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[33437:MainThread](2026-05-26 16:34:31,436) INFO - qlib.timer - [log.py:127] - Time cost: 0.004s | Init data Done


,fold,feature_start,feature_end,test_start,test_end,warm_start,warm_end
0,wf_2022,2011-03-11,2022-12-30,2022-01-01,2022-12-31,2021-10-07,2022-12-30
1,wf_2023_2024,2011-03-11,2024-12-31,2024-01-01,2024-12-31,2023-10-05,2024-12-31


In [3]:

rows = []
fit_rows = []
daily_manifest = []
for item in CANDIDATES:
    candidate = item["candidate"]
    targets = targets_by_candidate[candidate]
    for fold_name, spec in WALK_FORWARD_WINDOWS.items():
        fold_targets = fold_target_subset(targets, spec)
        if fold_targets.empty:
            rows.append({"candidate": candidate, "fold": fold_name, "status": "skipped_no_targets"})
            continue
        set_seed(SEED + int(item["complexity"]))
        fit = fit_state_objective_bc_actor_from_frame(
            train_frame,
            targets=fold_targets,
            profile=profile,
            train_config=BC_TRAIN_CONFIG,
            sample_weight_config=item["sample_weight_config"],
            sample_size=0,
            rng_seed=SEED,
        )
        fit_rows.append({
            "candidate": candidate,
            "fold": fold_name,
            "target_rows": int(len(fold_targets)),
            "bc_samples": int(len(fit.actions)),
            "after_accuracy": float(fit.metrics.get("after_accuracy", np.nan)),
            "after_loss": float(fit.metrics.get("after_loss", np.nan)),
            "weight_mean": float(fit.metrics.get("weight_mean", np.nan)) if "weight_mean" in fit.metrics else np.nan,
            "weight_max": float(fit.metrics.get("weight_max", np.nan)) if "weight_max" in fit.metrics else np.nan,
        })
        cache = fold_cache[fold_name]
        bt = backtest_profile(
            fit.actor,
            cache["feature_full"],
            cache["price_full"],
            position_cap_config=profile.position_cap,
            position_budget_config=profile.position_budget,
            execution_config=profile.execution,
            asset_idx=0,
            feature_columns=feature_columns,
            model_config=profile.model,
            mode_router_config=profile.mode_router,
            test_seg=cache["warm_seg"],
        )
        warm_daily = slice_warm_start_daily_path(
            portfolio_df=bt["portfolio"],
            price_series=cache["price_full"],
            eval_seg=spec["test"],
            position_history=bt.get("position_history", []),
            trade_signals=bt.get("trade_signals", []),
        )
        daily_path = CARD / f"{candidate}_{fold_name}_warm_start_official_daily_path.csv"
        warm_daily.to_csv(daily_path, index=False)
        metrics = daily_path_metrics(warm_daily)
        rows.append({
            "candidate": candidate,
            "fold": fold_name,
            "status": "evaluated",
            "requested_start": spec["test"][0],
            "requested_end": spec["test"][1],
            "warm_start": cache["warm_seg"][0],
            "warm_end": cache["warm_seg"][1],
            "official_warm_return": float(metrics["return"]),
            "official_warm_buyhold_return": float(metrics["buy_and_hold_return"]),
            "official_warm_relative_return": float(metrics["relative_return"]),
            "official_warm_max_dd": float(metrics["max_dd"]),
            "official_warm_avg_pos": float(metrics["avg_pos"]),
            "trades": int(bt.get("trades", 0)),
            "full_gate": bool(float(metrics["relative_return"]) >= TARGET_RELATIVE_RETURN and float(metrics["max_dd"]) <= TARGET_MAX_DD),
            "gate_loss": gate_loss(float(metrics["relative_return"]), float(metrics["max_dd"])),
            "daily_path": str(daily_path.relative_to(ROOT)),
        })
        daily_manifest.append({"candidate": candidate, "fold": fold_name, "path": str(daily_path.relative_to(ROOT))})

fit_metrics = pd.DataFrame(fit_rows).sort_values(["candidate", "fold"]).reset_index(drop=True)
warm_metrics = pd.DataFrame(rows).sort_values(["candidate", "fold"]).reset_index(drop=True)
fit_metrics.to_csv(CARD / "bc_fit_metrics.csv", index=False)
warm_metrics.to_csv(CARD / "warm_start_official_metrics.csv", index=False)
pd.DataFrame(daily_manifest).to_csv(CARD / "daily_manifest.csv", index=False)
display(fit_metrics)
display(warm_metrics)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[33437:MainThread](2026-05-26 16:34:37,547) WARNING - qlib.data - [data.py:665] - load calendar error: freq=day, future=True; return current calendar!


[33437:MainThread](2026-05-26 16:34:37,550) WARNING - qlib.data - [data.py:668] - You can get future calendar by referring to the following document: https://github.com/microsoft/qlib/blob/main/scripts/data_collector/contrib/README.md


[33437:MainThread](2026-05-26 16:34:37,559) WARNING - qlib.BaseExecutor - [executor.py:121] - `common_infra` is not set for <qlib.backtest.executor.SimulatorExecutor object at 0x138e73b50>


[33437:MainThread](2026-05-26 16:34:37,566) INFO - qlib.backtest caller - [__init__.py:93] - Create new exchange


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

,candidate,fold,target_rows,bc_samples,after_accuracy,after_loss,weight_mean,weight_max
0,base_plain_contract,wf_2022,253,253,0.833992,1.073453,NaN,NaN
1,base_plain_contract,wf_2023_2024,756,755,0.768212,1.146839,NaN,NaN
2,trend_plain_contract,wf_2022,253,253,0.833992,1.073453,NaN,NaN
3,trend_plain_contract,wf_2023_2024,756,755,0.754967,1.154651,NaN,NaN
4,trend_plus_slope_low_cap15_weighted,wf_2022,253,253,0.810277,1.091642,1.0,2.048482
5,trend_plus_slope_low_cap15_weighted,wf_2023_2024,756,755,0.737748,1.170573,1.0,4.298243


,candidate,fold,status,requested_start,requested_end,warm_start,warm_end,official_warm_return,official_warm_buyhold_return,official_warm_relative_return,official_warm_max_dd,official_warm_avg_pos,trades,full_gate,gate_loss,daily_path
0,base_plain_contract,wf_2022,evaluated,2022-01-01,2022-12-31,2021-10-07,2022-12-30,-18.689254,-86.588072,67.898818,20.198957,0.131701,42,False,51.989571,runtime/model_selection/card/sequence_trend_jo...
1,base_plain_contract,wf_2023_2024,evaluated,2024-01-01,2024-12-31,2023-10-05,2024-12-31,-0.493579,-0.356635,-0.136944,20.987807,0.159047,49,False,90.015010,runtime/model_selection/card/sequence_trend_jo...
2,trend_plain_contract,wf_2022,evaluated,2022-01-01,2022-12-31,2021-10-07,2022-12-30,-18.689254,-86.588072,67.898818,20.198957,0.131701,42,False,51.989571,runtime/model_selection/card/sequence_trend_jo...
3,trend_plain_contract,wf_2023_2024,evaluated,2024-01-01,2024-12-31,2023-10-05,2024-12-31,7.867928,-0.356635,8.224563,12.285728,0.147325,47,False,21.775437,runtime/model_selection/card/sequence_trend_jo...
4,trend_plus_slope_low_cap15_weighted,wf_2022,evaluated,2022-01-01,2022-12-31,2021-10-07,2022-12-30,-27.857539,-86.588072,58.730532,28.944800,0.134772,42,False,139.448000,runtime/model_selection/card/sequence_trend_jo...
5,trend_plus_slope_low_cap15_weighted,wf_2023_2024,evaluated,2024-01-01,2024-12-31,2023-10-05,2024-12-31,5.978095,-0.356635,6.334730,12.528079,0.140492,45,False,23.665270,runtime/model_selection/card/sequence_trend_jo...


In [4]:

prior_recon = pd.read_csv(RECON_CARD / "metric_reconciliation.csv")
prior_test = prior_recon[prior_recon["segment"].astype(str).eq("test")].copy()
prior_cols = [
    "candidate",
    "fold",
    "official_return",
    "official_max_dd",
    "calendar_relative_return",
    "tradable_relative_return",
    "calendar_full_gate",
    "tradable_full_gate",
]
comparison = warm_metrics.merge(prior_test[prior_cols], on=["candidate", "fold"], how="left", suffixes=("", "_prior_cold"))
comparison["warm_vs_cold_calendar_relative_delta"] = comparison["official_warm_relative_return"] - comparison["calendar_relative_return"]
comparison["warm_vs_cold_maxdd_delta"] = comparison["official_warm_max_dd"] - comparison["official_max_dd"]
comparison.to_csv(CARD / "warm_vs_prior_cold_comparison.csv", index=False)
display(comparison[[
    "candidate", "fold", "official_warm_relative_return", "official_warm_max_dd", "full_gate",
    "calendar_relative_return", "official_max_dd", "calendar_full_gate",
    "tradable_relative_return", "warm_vs_cold_calendar_relative_delta", "warm_vs_cold_maxdd_delta",
]])


,candidate,fold,official_warm_relative_return,official_warm_max_dd,full_gate,calendar_relative_return,official_max_dd,calendar_full_gate,tradable_relative_return,warm_vs_cold_calendar_relative_delta,warm_vs_cold_maxdd_delta
0,base_plain_contract,wf_2022,67.898818,20.198957,False,70.053568,18.396429,False,60.079764,-2.154750,1.802528
1,base_plain_contract,wf_2023_2024,-0.136944,20.987807,False,-5.455232,13.472617,False,34.140849,5.318288,7.515189
2,trend_plain_contract,wf_2022,67.898818,20.198957,False,70.053568,18.396429,False,60.079764,-2.154750,1.802528
3,trend_plain_contract,wf_2023_2024,8.224563,12.285728,False,-1.007555,12.897887,False,38.588527,9.232117,-0.612159
4,trend_plus_slope_low_cap15_weighted,wf_2022,58.730532,28.944800,False,67.727122,20.305636,False,57.753318,-8.996590,8.639164
5,trend_plus_slope_low_cap15_weighted,wf_2023_2024,6.334730,12.528079,False,-3.681043,12.392254,False,35.915039,10.015773,0.135825


In [5]:

evaluated = warm_metrics[warm_metrics["status"].eq("evaluated")].copy()
summary_rows = []
for candidate, group in evaluated.groupby("candidate", sort=True):
    group = group.sort_values("fold")
    row = {
        "candidate": candidate,
        "fold_count": int(len(group)),
        "full_gate_all": bool(group["full_gate"].all()),
        "full_gate_count": int(group["full_gate"].sum()),
        "gate_loss": float(group["gate_loss"].sum()),
        "min_relative_return": float(group["official_warm_relative_return"].min()),
        "max_max_dd": float(group["official_warm_max_dd"].max()),
        "mean_avg_pos": float(group["official_warm_avg_pos"].mean()),
        "total_trades": int(group["trades"].sum()),
    }
    for fold in WALK_FORWARD_WINDOWS:
        sub = group[group["fold"].eq(fold)]
        if not sub.empty:
            rec = sub.iloc[0]
            row[f"{fold}_relative_return"] = float(rec["official_warm_relative_return"])
            row[f"{fold}_max_dd"] = float(rec["official_warm_max_dd"])
            row[f"{fold}_avg_pos"] = float(rec["official_warm_avg_pos"])
            row[f"{fold}_full_gate"] = bool(rec["full_gate"])
    summary_rows.append(row)

candidate_summary = pd.DataFrame(summary_rows).sort_values(
    ["full_gate_all", "full_gate_count", "gate_loss", "candidate"],
    ascending=[False, False, True, True],
).reset_index(drop=True)
candidate_summary.to_csv(CARD / "candidate_summary.csv", index=False)
selected = candidate_summary.iloc[0].to_dict() if not candidate_summary.empty else {}
selected_candidate = str(selected.get("candidate", ""))
full_gate_count = int(candidate_summary["full_gate_all"].sum()) if not candidate_summary.empty else 0
if full_gate_count > 0:
    decision = "warm_start_official_candidate_ready_no_promotion"
    next_mechanism = "run_fixed_split_plus_wf2020_official_replay_and_actor_retention_review_before_any_reward_ppo"
    promotion_allowed = False
    ppo_allowed = False
else:
    decision = "warm_start_official_eval_no_full_gate_no_promotion"
    next_mechanism = "do_not_run_reward_ppo_from_sequence_trend_joint; use warm-start official replay as required gate and return to training-objective redesign"
    promotion_allowed = False
    ppo_allowed = False

selection = {
    "experiment": EXP,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "role": "diagnose",
    "profile": PROFILE_NAME,
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": ["sequence_trend_joint_bc_screen_v1", "sequence_trend_joint_metric_reconciliation_v1"],
    "closest_historical_success_case": "trend_participation_bc_warmstart_screen_v1 improved trend participation but retained drawdown tradeoffs",
    "closest_historical_failure_case": "sequence_trend_joint_metric_reconciliation_v1 found split/official mismatch and blocked PPO handoff",
    "why_not_duplicate": "This notebook reruns official qlib replay with warm-start state and then slices/rebases the requested evaluation window.",
    "changed_layer": ["model_selection", "harness_metric_diagnostic"],
    "target_relative_return_pp": TARGET_RELATIVE_RETURN,
    "target_max_dd_pct": TARGET_MAX_DD,
    "folds_evaluated": list(WALK_FORWARD_WINDOWS.keys()),
    "candidates_evaluated": [item["candidate"] for item in CANDIDATES],
    "selected_candidate": selected_candidate,
    "selected_summary": selected,
    "full_gate_candidate_count": full_gate_count,
    "promotion_allowed": promotion_allowed,
    "ppo_screen_allowed_next": ppo_allowed,
    "training_allowed_from_this_card": False,
    "actor_retention": "no actor bundle retained; diagnostic BC actors are regenerated by this notebook",
    "source_selection_decision": source_selection.get("decision"),
    "source_selected_candidate": source_selection.get("selected_candidate"),
    "metric_reconciliation_decision": recon_selection.get("decision"),
    "root_cause_checked": "official_replay_cold_start_vs_warm_start_sliced_metric",
    "next_mechanism": next_mechanism,
    "external_sources": EXTERNAL_SOURCES,
    "sidecars": [
        "warm_start_segments.csv",
        "bc_fit_metrics.csv",
        "warm_start_official_metrics.csv",
        "warm_vs_prior_cold_comparison.csv",
        "candidate_summary.csv",
        "daily_manifest.csv",
        "selection.json",
        "summary.json",
        "*_warm_start_official_daily_path.csv",
    ],
}
summary = {
    **selection,
    "candidate_summary": candidate_summary.to_dict("records"),
    "warm_start_official_metrics": warm_metrics.to_dict("records"),
    "comparison": comparison.to_dict("records"),
    "conclusion": [],
}
for row in candidate_summary.to_dict("records"):
    summary["conclusion"].append(
        f"{row['candidate']}: min warm-start official relative={row['min_relative_return']:.2f}pp, "
        f"max MaxDD={row['max_max_dd']:.2f}%, full_gate_all={bool(row['full_gate_all'])}, gate_loss={row['gate_loss']:.2f}."
    )
if full_gate_count > 0:
    summary["conclusion"].append("At least one candidate passes the two tested warm-start official folds, but no promotion because fixed-split and wf_2020 replay are still required.")
else:
    summary["conclusion"].append("Warm-start official replay does not rescue the sequence-trend joint BC family under the +30pp / <=15% MaxDD full gate.")
summary["conclusion"].append("Do not hand this card to reward/PPO; keep warm-start official daily paths as the required evaluation source for future BC screens.")
write_json(CARD / "selection.json", selection)
write_json(CARD / "summary.json", summary)
display(candidate_summary)
print(json.dumps(json_safe(selection), indent=2, allow_nan=False))


,candidate,fold_count,full_gate_all,full_gate_count,gate_loss,min_relative_return,max_max_dd,mean_avg_pos,total_trades,wf_2022_relative_return,wf_2022_max_dd,wf_2022_avg_pos,wf_2022_full_gate,wf_2023_2024_relative_return,wf_2023_2024_max_dd,wf_2023_2024_avg_pos,wf_2023_2024_full_gate
0,trend_plain_contract,2,False,0,73.765009,8.224563,20.198957,0.139513,89,67.898818,20.198957,0.131701,False,8.224563,12.285728,0.147325,False
1,base_plain_contract,2,False,0,142.004581,-0.136944,20.987807,0.145374,91,67.898818,20.198957,0.131701,False,-0.136944,20.987807,0.159047,False
2,trend_plus_slope_low_cap15_weighted,2,False,0,163.113270,6.334730,28.944800,0.137632,87,58.730532,28.944800,0.134772,False,6.334730,12.528079,0.140492,False


{
  "experiment": "sequence_trend_joint_warm_start_official_eval_v1",
  "created_at": "2026-05-26T08:35:28.218945+00:00",
  "decision": "warm_start_official_eval_no_full_gate_no_promotion",
  "role": "diagnose",
  "profile": "sparse_release_bc_ppo_screen_v1",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": [
    "sequence_trend_joint_bc_screen_v1",
    "sequence_trend_joint_metric_reconciliation_v1"
  ],
  "closest_historical_success_case": "trend_participation_bc_warmstart_screen_v1 improved trend participation but retained drawdown tradeoffs",
  "closest_historical_failure_case": "sequence_trend_joint_metric_reconciliation_v1 found split/official mismatch and blocked PPO handoff",
  "why_not_duplicate": "This notebook reruns official qlib replay with warm-start state and then slices/rebases the requested evaluation window.",
  "changed_layer": [
    "model_selection",
    "harness_metric_diagnostic"
  ],
  "target_relative_return_pp": 

## Conclusion

The executable cells above write the run-local evidence. The authoritative single-run conclusion is in `summary.json` and this notebook's final displayed selection. The promotion decision is intentionally conservative: this diagnostic only tests whether warm-start official qlib replay rescues the prior sequence/trend joint BC candidates; it does not retain actors or authorize PPO.